# Generalized DBSCAN clustering

Clusters all 445 feature names together, without filtering by product type.

Like the other clustering notebooks, this one loads the shared similarity matrix from `data/similarity.npz` (saved by `3_similarity.ipynb`). This keeps the input fixed, so result differences come from the algorithm itself.

**Method**:
1. Convert similarity to distance (`distance = 1 - similarity`).
2. Try a compact grid of DBSCAN parameters (`eps`, `min_samples`).
3. Select the configuration with the best silhouette score, counting noise points as singleton clusters.
4. Inspect clusters and visualize with t-SNE.
5. Check parameter sensitivity (k-distance plot, grid heatmaps).

In [1]:
from collections import Counter

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display
from sklearn.cluster import DBSCAN
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score

EPS_VALUES = np.round(np.arange(0.20, 0.82, 0.02), 2)
MIN_SAMPLES_VALUES = [2, 3, 4, 5, 6, 8]
SEED = 0
HIGHLIGHT_GROUPS = 8

with np.load("data/similarity.npz") as data:
    similarity = data["similarity"]
    feature_names = data["titles"].tolist()

n = len(feature_names)
assert similarity.shape == (n, n)
assert np.allclose(similarity, similarity.T), "similarity must be symmetric"

distance_matrix = 1 - similarity
distance_matrix = np.clip(distance_matrix, 0, 1)
np.fill_diagonal(distance_matrix, 0)

print(f"Loaded {n} feature names, matrix {similarity.shape}")

Loaded 445 attributes, matrix (445, 445)


## 1. Parameter search

For each `(eps, min_samples)` pair, we fit DBSCAN on the precomputed distance matrix.

Parameters are selected by the silhouette score computed on all feature names, with each noise point treated as its own singleton cluster (silhouette 0). This prevents a configuration from scoring well by discarding difficult feature names as noise. Ties are broken by higher coverage, then fewer clusters. Both methods search the same `min_samples` range; DBSCAN additionally searches `eps` and HDBSCAN `min_cluster_size`. HDBSCAN has no `eps` parameter.

The silhouette on non-noise points only is reported for reference but is not used for selection.

In [2]:
def noise_as_singletons(labels):
    relabeled = labels.copy()
    noise_indices = np.where(relabeled == -1)[0]
    start = relabeled.max() + 1
    relabeled[noise_indices] = np.arange(start, start + len(noise_indices))
    return relabeled


def evaluate(labels, D):
    is_noise = labels == -1
    n_noise = int(is_noise.sum())
    n_clusters = len(set(labels[~is_noise].tolist()))
    if n_clusters < 2:
        return None

    singleton_labels = noise_as_singletons(labels)
    if len(set(singleton_labels.tolist())) >= len(labels):
        return None
    score = silhouette_score(D, singleton_labels, metric="precomputed")

    non_noise_labels = labels[~is_noise]
    if n_clusters < len(non_noise_labels):
        silhouette_non_noise = silhouette_score(
            D[np.ix_(~is_noise, ~is_noise)], non_noise_labels, metric="precomputed"
        )
    else:
        silhouette_non_noise = float("nan")

    return {
        "score": float(score),
        "silhouette_non_noise": float(silhouette_non_noise),
        "coverage": float((~is_noise).mean()),
        "n_clusters": int(n_clusters),
        "n_noise": n_noise,
    }


def format_row(row, param_names):
    params = ", ".join(f"{name}={row[name]:g}" for name in param_names)
    return (
        f"{params} | "
        f"score={row['score']:.4f}, silhouette_non_noise={row['silhouette_non_noise']:.4f}, "
        f"clusters={row['n_clusters']}, coverage={100 * row['coverage']:.1f}%, noise={row['n_noise']}"
    )


def warn_if_at_grid_edge(best, grids):
    for name, grid in grids.items():
        if np.isclose(best[name], min(grid)) or np.isclose(best[name], max(grid)):
            print(f"Warning ({name}={best[name]:g}): Selected value is at the edge of the grid - consider extending the grid.")


PARAM_GRIDS = {"eps": EPS_VALUES, "min_samples": MIN_SAMPLES_VALUES}

candidates = []
for eps in EPS_VALUES:
    for min_samples in MIN_SAMPLES_VALUES:
        model = DBSCAN(eps=float(eps), min_samples=min_samples, metric="precomputed")
        labels = model.fit_predict(distance_matrix)
        evaluation = evaluate(labels, distance_matrix)
        if evaluation is None:
            continue
        candidates.append({"eps": float(eps), "min_samples": int(min_samples), **evaluation})

if not candidates:
    raise ValueError("No valid DBSCAN configuration produced at least 2 non-noise clusters.")

candidates = sorted(candidates, key=lambda r: (-r["score"], -r["coverage"], r["n_clusters"]))
best = candidates[0]

print("Top 10 parameter sets (score = silhouette with noise as singletons):")
for row in candidates[:10]:
    print(format_row(row, PARAM_GRIDS))

print("\nSelected configuration:")
print(format_row(best, PARAM_GRIDS))
warn_if_at_grid_edge(best, PARAM_GRIDS)

Top 10 parameter sets (score = silhouette with noise as singletons):
eps=0.4, min_samples=2 | score=0.1380, silhouette_non_noise=0.4755, clusters=59, coverage=37.1%, noise=280
eps=0.38, min_samples=2 | score=0.1323, silhouette_non_noise=0.4946, clusters=56, coverage=34.4%, noise=292
eps=0.42, min_samples=2 | score=0.1265, silhouette_non_noise=0.4172, clusters=61, coverage=41.8%, noise=259
eps=0.36, min_samples=2 | score=0.1225, silhouette_non_noise=0.5323, clusters=50, coverage=29.7%, noise=313
eps=0.44, min_samples=2 | score=0.1220, silhouette_non_noise=0.3957, clusters=59, coverage=44.7%, noise=246
eps=0.46, min_samples=2 | score=0.1180, silhouette_non_noise=0.3604, clusters=58, coverage=48.3%, noise=230
eps=0.5, min_samples=2 | score=0.1163, silhouette_non_noise=0.3027, clusters=60, coverage=56.2%, noise=195
eps=0.34, min_samples=2 | score=0.1108, silhouette_non_noise=0.5663, clusters=43, coverage=24.9%, noise=334
eps=0.48, min_samples=2 | score=0.1094, silhouette_non_noise=0.3354, 

## 2. Fit DBSCAN and inspect groups

In [3]:
final_model = DBSCAN(eps=best["eps"], min_samples=best["min_samples"], metric="precomputed")
labels = final_model.fit_predict(distance_matrix)

is_noise = labels == -1
cluster_labels = [label for label in sorted(set(labels.tolist())) if label != -1]
sizes = Counter(labels)

print(
    f"Clusters (excluding noise): {len(cluster_labels)} | "
    f"noise points: {sizes.get(-1, 0)} / {len(labels)} ({100 * sizes.get(-1, 0) / len(labels):.1f}%)"
)
print(f"Cluster sizes (largest first): {[size for label, size in sizes.most_common() if label != -1]}")

members_by_cluster = {
    label: [feature_names[index] for index, assigned_label in enumerate(labels) if assigned_label == label]
    for label in cluster_labels
}

for label, members in sorted(members_by_cluster.items(), key=lambda item: (-len(item[1]), item[0])):
    if len(members) > 1:
        print(f"\nCluster {label} ({len(members)} feature names)")
        for member in members:
            print(f"  - {member}")

Clusters (excluding noise): 59 | noise points: 280 / 445 (62.9%)
Cluster sizes (largest first): [9, 6, 6, 6, 5, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]

Cluster 2 (9 attributes)
  - Windproof Front Panel
  - Windproof protection
  - Windproof softshell 3-layer panels
  - Windtex Softshell 3-layer fabric
  - Windproof front panel
  - Windproof softshell fabric
  - Windproof front panels
  - Deep winter windproof protection
  - Windproof

Cluster 15 (6 attributes)
  - Highly breathable
  - Breathable
  - Extremely breathable
  - Super breathable
  - Extremely breathable 40K (40,000 g/m²/24h)
  - Breathable fabric

Cluster 17 (6 attributes)
  - Two-way YKK® Vislon zip
  - YKK 2-way zipper
  - Two-way YKK® Vislon AquaGuard zip
  - 2-way YKK front zipper
  - YKK Vislon 2-way zipper
  - YKK 2-way waterproof zipper

Cluster 33 (6 attributes)
  - Convenient rear pockets
  -

## 3. Cluster visualization

t-SNE projects precomputed distances (`1 - similarity`) to two dimensions for display only.
The largest groups receive highlight colors, smaller groups are grey, and noise points are hollow markers.

In [4]:
INK = "#2b2b2b"
GRAY = "#d7d7d7"
LIGHT_GRAY = "#c9c8c0"
HIGHLIGHT_PALETTE = ["#cc1f1a", "#005f73", "#0a9396", "#ee9b00", "#ae2012", "#6c757d", "#9b2226", "#3a86ff"]

perplexity = min(30, len(feature_names) - 1)
coordinates = TSNE(
    n_components=2,
    metric="precomputed",
    init="random",
    random_state=SEED,
    perplexity=perplexity,
).fit_transform(distance_matrix)

multi = [label for label in cluster_labels if sizes[label] > 1]
ranked_groups = sorted(multi, key=lambda label: -sizes[label])
highlighted_groups = ranked_groups[:HIGHLIGHT_GROUPS]

def group_name(label):
    indices = np.where(labels == label)[0]
    within_group_similarity = similarity[np.ix_(indices, indices)]
    medoid_index = indices[within_group_similarity.sum(axis=1).argmax()]
    return feature_names[medoid_index]

group_names = {label: group_name(label) for label in multi}

def make_trace(indices, name, color, custom_group_names, hollow=False, size=9):
    if len(indices) == 0:
        return None
    return go.Scatter(
        x=coordinates[indices, 0],
        y=coordinates[indices, 1],
        mode="markers",
        name=name,
        marker=dict(
            size=size,
            color="white" if hollow else color,
            line=dict(width=1.5 if hollow else 1, color=color if hollow else "white"),
        ),
        text=[feature_names[index] for index in indices],
        customdata=[custom_group_names[index] for index in indices],
        hovertemplate="<b>%{text}</b><br>group: %{customdata}<extra></extra>",
    )

custom_group_names = []
for label in labels:
    if label == -1:
        custom_group_names.append("Noise")
    else:
        custom_group_names.append(group_names.get(label, f"Cluster {label}"))

noise_indices = np.where(labels == -1)[0]
other_group_indices = np.array(
    [index for index, label in enumerate(labels) if label != -1 and sizes[label] > 1 and label not in highlighted_groups],
    dtype=int,
)

figure = go.Figure()

trace = make_trace(noise_indices, f"Noise ({len(noise_indices)})", LIGHT_GRAY, custom_group_names, hollow=True, size=7)
if trace is not None:
    figure.add_trace(trace)

trace = make_trace(
    other_group_indices,
    f"Other groups ({max(len(ranked_groups) - len(highlighted_groups), 0)})",
    GRAY,
    custom_group_names,
    size=7,
)
if trace is not None:
    figure.add_trace(trace)

for color, label in zip(HIGHLIGHT_PALETTE, highlighted_groups):
    indices = np.where(labels == label)[0]
    trace = make_trace(indices, f"{group_names[label]} ({sizes[label]})", color, custom_group_names, size=10)
    if trace is not None:
        figure.add_trace(trace)

figure.update_layout(
    title=dict(text="Feature-name clusters: DBSCAN (t-SNE)", x=0, font=dict(color=INK)),
    template="simple_white",
    width=1000,
    height=720,
    xaxis=dict(visible=False),
    yaxis=dict(visible=False),
    legend=dict(title="Largest groups", font=dict(color=INK), itemsizing="constant"),
    hoverlabel=dict(bgcolor="white", font_color=INK),
)
try:
    figure.show(renderer="notebook_connected")
except ValueError:
    display(HTML(figure.to_html(full_html=False, include_plotlyjs=True)))

## 5. Evaluation

### 5.1 Internal quality

DBSCAN marks some points as noise (`-1`). The selection score treats each noise point as its own singleton cluster (silhouette 0), so discarding feature names as noise does not inflate it. The silhouette on non-noise points only is shown for reference.

In [5]:
metrics = evaluate(labels, distance_matrix)

if metrics is None:
    print("Silhouette score not defined (need at least 2 non-noise clusters).")
else:
    print(f"Silhouette score (noise as singletons, selection score): {metrics['score']:.4f}")
    print(f"Silhouette score (non-noise only, reported only): {metrics['silhouette_non_noise']:.4f}")
    print(f"Clusters (excluding noise): {metrics['n_clusters']}")
    print(f"Coverage: {100 * metrics['coverage']:.1f}%")
    print(f"Noise points: {metrics['n_noise']} / {len(labels)}")

Silhouette score (noise as singletons, selection score): 0.1380
Silhouette score (non-noise only, reported only): 0.4755
Clusters (excluding noise): 59
Coverage: 37.1%
Noise points: 280 / 445


### 5.2 Parameter selection

These plots show how sensitive the result is to the chosen parameters, using the results already computed in the parameter search (no refitting).

- **k-distance plot** sorts every feature name's distance to its `min_samples`-th nearest neighbour (excluding itself). Points left of where the curve bends sharply upward lie in dense regions; the dashed line is the selected `eps`. If the line sits near the bend, `eps` separates dense feature names from sparse ones; feature names above the line cannot be core points.
- **Heatmaps** show the selection score, coverage and number of clusters for every configuration in the grid, with the same colour map in both notebooks. The red square marks the selected configuration. Blank cells produced fewer than 2 clusters. A selected cell inside a broad bright area means the choice is stable; an isolated peak, or one at the grid edge, means it depends on the exact values.
- **Sensitivity summary** lists the score and number of clusters one grid step away from the selected configuration in each direction. Small changes in score but large changes in cluster count mean the score alone does not pin down the granularity.

_Placeholder: DBSCAN finds N clusters with the selected parameters, compared with k = … chosen for spectral clustering in `5_spectral.ipynb`._

In [6]:
m = best["min_samples"]
nth = f"{m}{ {1: 'st', 2: 'nd', 3: 'rd'}.get(m if m < 20 else m % 10, 'th') }"
# Column 0 of each sorted row is the point itself (distance 0), so column m is the m-th neighbour.
k_distances = np.sort(np.sort(distance_matrix, axis=1)[:, m])

figure = go.Figure(
    go.Scatter(
        x=np.arange(1, len(k_distances) + 1), y=k_distances, mode="lines",
        line=dict(color="#2b2b2b", width=2), name=f"distance to {nth} neighbour",
        hovertemplate="rank %{x}<br>distance %{y:.3f}<extra></extra>",
    )
)
figure.add_hline(
    y=best["eps"], line=dict(color="#cc1f1a", dash="dash", width=2), opacity=1,
    annotation_text=f"selected eps = {best['eps']:.2f}", annotation_position="top left",
)
figure.update_layout(
    title=dict(text=f"k-distance plot (m = min_samples = {m})", x=0),
    template="simple_white", width=1000, height=450,
    xaxis_title="Feature names, sorted by distance", yaxis_title=f"Distance to {nth} nearest neighbour",
)
try:
    figure.show(renderer="notebook_connected")
except ValueError:
    display(HTML(figure.to_html(full_html=False, include_plotlyjs=True)))

print(f"{(k_distances <= best['eps']).mean():.1%} of feature names have their {nth} neighbour within eps = {best['eps']:.2f}")

14.8% of attributes have their 2nd neighbour within eps = 0.40


In [7]:
SELECTED_COLOR = "#cc1f1a"
HEATMAP_COLORSCALE = "Viridis"
HEATMAP_METRICS = {
    "score": ("Score (silhouette, noise as singletons)", ".3f"),
    "coverage": ("Coverage (%)", ".1f"),
    "n_clusters": ("Clusters", ".0f"),
}


def grid_table(candidates, x_name, x_values, y_name, y_values):
    lookup = {(row[x_name], row[y_name]): row for row in candidates}
    table = {metric: np.full((len(y_values), len(x_values)), np.nan) for metric in HEATMAP_METRICS}
    for j, x in enumerate(x_values):
        for i, y in enumerate(y_values):
            row = lookup.get((x, y))
            if row is None:
                continue
            for metric in HEATMAP_METRICS:
                table[metric][i, j] = 100 * row[metric] if metric == "coverage" else row[metric]
    return table


def plot_param_heatmaps(table, x_name, x_values, y_name, y_values, best, method):
    x_labels = [f"{x:g}" for x in x_values]
    y_labels = [f"{y:g}" for y in y_values]
    figure = make_subplots(
        rows=len(HEATMAP_METRICS), cols=1,
        subplot_titles=[title for title, _ in HEATMAP_METRICS.values()],
        vertical_spacing=0.1,
    )
    for row_index, (metric, (title, fmt)) in enumerate(HEATMAP_METRICS.items(), start=1):
        domain = figure.get_subplot(row_index, 1).yaxis.domain
        figure.add_trace(
            go.Heatmap(
                z=table[metric], x=x_labels, y=y_labels,
                colorscale=HEATMAP_COLORSCALE, hoverongaps=False,
                colorbar=dict(len=domain[1] - domain[0], y=(domain[0] + domain[1]) / 2, thickness=12),
                hovertemplate=f"{x_name}=%{{x}}<br>{y_name}=%{{y}}<br>{title}: %{{z:{fmt}}}<extra></extra>",
            ),
            row=row_index, col=1,
        )
        figure.add_trace(
            go.Scatter(
                x=[f"{best[x_name]:g}"], y=[f"{best[y_name]:g}"], mode="markers",
                marker=dict(symbol="square-open", size=18, color=SELECTED_COLOR, line=dict(width=3)),
                name="Selected", showlegend=row_index == 1, hoverinfo="skip",
            ),
            row=row_index, col=1,
        )
        figure.update_xaxes(
            title_text=x_name if row_index == len(HEATMAP_METRICS) else None,
            type="category", tickangle=-45 if len(x_values) > 12 else 0, row=row_index, col=1,
        )
        figure.update_yaxes(title_text=y_name, type="category", row=row_index, col=1)
    figure.update_layout(
        title=dict(text=f"{method} parameter grid (blank = fewer than 2 clusters or not evaluated)", x=0),
        template="simple_white", width=1000, height=1000,
        legend=dict(orientation="h", x=1, xanchor="right", y=1.04),
    )
    show(figure)


def print_sensitivity(candidates, best, grids):
    names = list(grids)
    lookup = {tuple(row[name] for name in names): row for row in candidates}
    print(f"Selected: {format_row(best, grids)}")
    print("\nOne grid step away (score change vs selected):")
    for name, grid in grids.items():
        grid = list(grid)
        index = int(np.argmin([abs(value - best[name]) for value in grid]))
        for step in (-1, 1):
            if not 0 <= index + step < len(grid):
                print(f"  {name}: no {'lower' if step < 0 else 'higher'} value in grid")
                continue
            key = tuple(grid[index + step] if other == name else best[other] for other in names)
            params = ", ".join(f"{other}={value:g}" for other, value in zip(names, key))
            row = lookup.get(key)
            if row is None:
                print(f"  {params} | fewer than 2 clusters or not evaluated")
            else:
                print(
                    f"  {params} | score={row['score']:.4f} ({row['score'] - best['score']:+.4f}), "
                    f"clusters={row['n_clusters']}"
                )


def show(figure):
    try:
        figure.show(renderer="notebook_connected")
    except ValueError:
        display(HTML(figure.to_html(full_html=False, include_plotlyjs=True)))


grids = {"eps": EPS_VALUES.tolist(), "min_samples": MIN_SAMPLES_VALUES}
table = grid_table(candidates, "eps", grids["eps"], "min_samples", grids["min_samples"])
plot_param_heatmaps(table, "eps", grids["eps"], "min_samples", grids["min_samples"], best, "DBSCAN")
print_sensitivity(candidates, best, grids)

Selected: eps=0.4, min_samples=2 | score=0.1380, silhouette_non_noise=0.4755, clusters=59, coverage=37.1%, noise=280

One grid step away (score change vs selected):
  eps=0.38, min_samples=2 | score=0.1323 (-0.0058), clusters=56
  eps=0.42, min_samples=2 | score=0.1265 (-0.0115), clusters=61
  min_samples: no lower value in grid
  eps=0.4, min_samples=3 | score=0.0622 (-0.0759), clusters=24
